# Práctica guiada: CNN y Transfer Learning con CIFAR-10

**Nivel:** intermedio  
**Duración sugerida:** 3–4 horas  
**Dataset:** CIFAR-10 (imágenes RGB de 32×32, 10 clases)  
**Comparación:** CNN propia desde cero vs. ResNet18 preentrenada  
**Modalidad:** completa los `TODO` y contrasta después con el solucionario.

## Resultados de aprendizaje

Al finalizar podrás:

1. preparar imágenes RGB con aumento de datos y normalización;
2. construir una CNN con convolución, Batch Normalization, pooling y Dropout;
3. entrenar y evaluar modelos multiclase con métricas por clase;
4. cargar una ResNet18 con pesos de ImageNet;
5. congelar el extractor, entrenar una nueva cabeza y realizar *fine-tuning*;
6. comparar exactitud, tiempo y cantidad de parámetros entre ambos enfoques.

> **Entorno recomendado:** Google Colab con GPU. Si falta alguna dependencia, ejecuta una vez:  
> `%pip install torch torchvision scikit-learn matplotlib`

El notebook usa `FAST_MODE=True` por defecto para reducir el tiempo de clase. Cambia a `False` para utilizar todo CIFAR-10 y más épocas. La primera ejecución descargará CIFAR-10 y los pesos preentrenados de ResNet18.


## Bloque A — Preparación del experimento


## Ejercicio 1 — Entorno reproducible y modos de ejecución

**Objetivo:** preparar un experimento que funcione en CPU, CUDA o MPS.

1. Importa PyTorch, torchvision, NumPy, Matplotlib y las utilidades indicadas.
2. Fija `SEED = 42` para Python, NumPy y PyTorch.
3. Detecta el dispositivo disponible.
4. Mantén `FAST_MODE=True` para la práctica; el modo completo usa más datos y épocas.

**Comprobación:** imprime las versiones, el dispositivo y la configuración de épocas.


In [ ]:
import copy
import random
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
import torchvision
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import transforms
from torchvision.datasets import CIFAR10
from torchvision.models import ResNet18_Weights, resnet18
from sklearn.model_selection import train_test_split
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

SEED = 42
FAST_MODE = True

def set_seed(seed=SEED):
    # TODO: fija las semillas de random, NumPy, torch y todas las GPU CUDA.
    pass

set_seed()

# TODO: selecciona cuda, luego mps y finalmente cpu.
DEVICE = None

BASELINE_EPOCHS = 6 if FAST_MODE else 15
HEAD_EPOCHS = 3 if FAST_MODE else 5
FINETUNE_EPOCHS = 2 if FAST_MODE else 5
NUM_WORKERS = 2 if DEVICE.type == "cuda" else 0

print("PyTorch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("Dispositivo:", DEVICE)
print("Épocas:", BASELINE_EPOCHS, HEAD_EPOCHS, FINETUNE_EPOCHS)


## Ejercicio 2 — Descargar y reconocer CIFAR-10

**Objetivo:** conocer el problema de clasificación.

1. Descarga los conjuntos oficiales de entrenamiento y prueba sin transformaciones.
2. Obtén los nombres de las 10 clases.
3. Imprime cantidad de imágenes, tamaño y modo de color.
4. Muestra una imagen de cada clase.

**Pregunta breve:** ¿qué hace más difícil CIFAR-10 que el dataset `digits` de 8×8 usado anteriormente?


In [ ]:
DATA_DIR = Path("./data")

# TODO: descarga CIFAR-10 de entrenamiento y prueba sin transformaciones.
raw_train = None
raw_test = None
CLASS_NAMES = None

print("Entrenamiento:", None)
print("Prueba:", None)
print("Clases:", CLASS_NAMES)

# TODO: muestra una imagen de cada clase en una cuadrícula 2x5.
fig, axes = plt.subplots(2, 5, figsize=(11, 5))
for class_id, ax in enumerate(axes.ravel()):
    pass
plt.tight_layout()
plt.show()


## Ejercicio 3 — Crear particiones estratificadas

**Objetivo:** usar exactamente las mismas imágenes al comparar los dos modelos.

1. En modo rápido, selecciona 15 000 imágenes del conjunto oficial de entrenamiento y 3 000 de prueba.
2. Divide las 15 000 en 12 000 para entrenamiento y 3 000 para validación.
3. En modo completo, usa 45 000/5 000/10 000.
4. Conserva la proporción de clases mediante `stratify`.

**Comprobación:** muestra tamaños y conteos por clase.


In [ ]:
train_targets = np.asarray(raw_train.targets)
test_targets = np.asarray(raw_test.targets)
all_train_indices = np.arange(len(raw_train))
all_test_indices = np.arange(len(raw_test))

if FAST_MODE:
    # TODO: selecciona 15 000 índices estratificados y 3 000 de prueba.
    selected_train_indices = None
    test_indices = None
    val_size = 3_000
else:
    selected_train_indices = all_train_indices
    test_indices = all_test_indices
    val_size = 5_000

# TODO: divide selected_train_indices en entrenamiento y validación.
train_indices, val_indices = None, None

print("Tamaños:", len(train_indices), len(val_indices), len(test_indices))
print("Train por clase:", None)
print("Val por clase:", None)
print("Test por clase:", None)


## Ejercicio 4 — Transformaciones para la CNN propia

**Objetivo:** diferenciar aumento de datos y normalización.

Para entrenamiento usa:

`RandomCrop(32, padding=4) → RandomHorizontalFlip → ToTensor → Normalize`

Para validación/prueba usa únicamente `ToTensor → Normalize`. Emplea la media y desviación estándar de CIFAR-10 indicadas en la celda.

1. Implementa ambas tuberías.
2. Define `denormalize` para visualizar tensores correctamente.
3. Muestra dos aumentos aleatorios de la misma imagen.


In [ ]:
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2470, 0.2435, 0.2616)

# TODO: crea train_transform_cnn y eval_transform_cnn.
train_transform_cnn = None
eval_transform_cnn = None

def denormalize(images, mean, std):
    # TODO: revierte channel-wise: x * std + mean y limita a [0, 1].
    pass

pil_image, _ = raw_train[int(train_indices[0])]

# TODO: aplica dos veces el aumento y visualiza ambos resultados.


## Ejercicio 5 — Datasets, DataLoaders y lotes RGB

**Objetivo:** construir cargadores reutilizando los índices anteriores.

1. Crea instancias de CIFAR-10 con transformaciones de entrenamiento y evaluación.
2. Envuelve cada una en `Subset` usando los índices correspondientes.
3. Crea `DataLoader` con lote 128 y barajado solo en entrenamiento.
4. Verifica que un lote tenga forma `(N, 3, 32, 32)`.


In [ ]:
# TODO: crea las tres vistas del dataset con sus transformaciones.
train_cnn_base = None
eval_train_cnn_base = None
test_cnn_base = None

# TODO: crea Subset para train, val y test.
train_cnn_ds = None
val_cnn_ds = None
test_cnn_ds = None

BATCH_SIZE = 128

def make_loader(dataset, shuffle):
    # TODO: configura batch_size, shuffle, generator, workers y pin_memory.
    pass

train_cnn_loader = make_loader(train_cnn_ds, shuffle=True)
val_cnn_loader = make_loader(val_cnn_ds, shuffle=False)
test_cnn_loader = make_loader(test_cnn_ds, shuffle=False)

images_batch, labels_batch = next(iter(train_cnn_loader))
print("Imágenes:", None)
print("Etiquetas:", None)


## Bloque B — CNN entrenada desde cero


## Ejercicio 6 — Observar convolución y pooling

**Objetivo:** comprobar cómo cambian canales y dimensiones espaciales.

Sobre cuatro imágenes:

1. aplica `Conv2d(3, 16, kernel_size=3, padding=1)` y ReLU;
2. aplica `MaxPool2d(2)`;
3. imprime las formas de entrada, mapas de características y salida del pooling.

Explica por qué se pasa de `(4,3,32,32)` a `(4,16,32,32)` y luego a `(4,16,16,16)`.


In [ ]:
sample_images = images_batch[:4].to(DEVICE)

# TODO: crea conv y pool; aplica conv -> ReLU -> pool sin gradientes.
demo_conv = None
demo_pool = None
feature_maps = None
pooled_maps = None

print("Entrada:", None)
print("Convolución:", None)
print("Pooling:", None)


## Ejercicio 7 — Construir una CNN con bloques convolucionales

**Objetivo:** diseñar una arquitectura más sólida que la CNN introductoria.

Crea `ConvBlock` con dos secuencias `Conv2d → BatchNorm2d → ReLU` y un `MaxPool2d(2)`. Después crea `CifarCNN`:

`3→32 → 32→64 → 64→128 → AdaptiveAvgPool(1) → Flatten → Dropout(0.3) → Linear(128,10)`

Usa `bias=False` en las convoluciones porque Batch Normalization ya incorpora un desplazamiento aprendible.


In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        # TODO: dos conv-bn-relu y un maxpool.
        self.block = None

    def forward(self, x):
        pass


class CifarCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        # TODO: tres ConvBlock, AdaptiveAvgPool y clasificador.
        self.features = None
        self.pool = None
        self.classifier = None

    def forward(self, x):
        pass

scratch_model = CifarCNN().to(DEVICE)
print(scratch_model)


## Ejercicio 8 — Trazar formas y contar parámetros

**Objetivo:** verificar la arquitectura antes de entrenarla.

1. Pasa cuatro imágenes por cada bloque e imprime las formas.
2. Completa el paso por pooling y clasificador.
3. Implementa funciones para contar parámetros totales y entrenables.
4. Verifica que la salida final sea `(4, 10)`.


In [ ]:
def count_parameters(model, trainable_only=False):
    # TODO: suma numel(); si trainable_only, filtra requires_grad.
    pass

scratch_model.eval()
with torch.no_grad():
    x = sample_images
    # TODO: recorre scratch_model.features e imprime cada forma.
    # TODO: aplica pool y classifier.
    scratch_logits = None

print("Logits:", None)
print("Parámetros totales:", None)
print("Parámetros entrenables:", None)


## Ejercicio 9 — Ciclos de entrenamiento con mejor modelo

**Objetivo:** crear funciones reutilizables para los dos enfoques.

Completa `run_epoch` y `fit` para:

- alternar correctamente `train()` y `eval()`;
- calcular pérdida y exactitud ponderadas por ejemplos;
- mantener BatchNorm en modo evaluación durante transferencia;
- aplicar un scheduler al final de cada época;
- conservar los pesos con mayor exactitud de validación;
- medir el tiempo total.


In [ ]:
def set_batchnorm_eval(module):
    # TODO: si module es BatchNorm, activa eval().
    pass


def run_epoch(model, loader, loss_fn, optimizer=None, freeze_bn=False):
    # TODO: implementa entrenamiento/evaluación y devuelve loss + accuracy.
    pass


def fit(
    model,
    train_loader,
    val_loader,
    loss_fn,
    optimizer,
    epochs,
    scheduler=None,
    freeze_bn=False,
):
    # TODO: guarda historial, mejor state_dict y tiempo transcurrido.
    pass


## Ejercicio 10 — Entrenar la CNN desde cero

**Objetivo:** establecer el modelo de referencia.

1. Reinicia la semilla e instancia una CNN nueva.
2. Usa `CrossEntropyLoss`, `AdamW(lr=1e-3, weight_decay=1e-4)` y `CosineAnnealingLR`.
3. Entrena durante `BASELINE_EPOCHS`.
4. Grafica pérdida y exactitud de entrenamiento/validación.

**Pregunta breve:** ¿las curvas sugieren subajuste, sobreajuste o un ajuste razonable?


In [ ]:
def plot_history(history, title):
    # TODO: crea dos gráficos: pérdida y accuracy.
    pass

set_seed()

# TODO: crea modelo, pérdida, AdamW, scheduler y ejecuta fit.
scratch_model = None
loss_fn = None
scratch_optimizer = None
scratch_scheduler = None
history_scratch, time_scratch = None, None

plot_history(history_scratch, "CNN desde cero")


## Ejercicio 11 — Evaluar la CNN por clase

**Objetivo:** identificar fortalezas y debilidades de la línea base.

1. Implementa `collect_predictions` sin gradientes.
2. Calcula pérdida y exactitud de prueba.
3. Muestra matriz de confusión y `classification_report`.
4. Calcula la exactitud de cada clase.

**Pregunta breve:** ¿qué clases confunde más y qué semejanza visual podría explicarlo?


In [ ]:
def collect_predictions(model, loader):
    # TODO: devuelve arrays NumPy con etiquetas reales y predichas.
    pass


def per_class_accuracy(y_true, y_pred, class_names):
    # TODO: devuelve un diccionario clase -> accuracy.
    pass

# TODO: evalúa, predice y visualiza las métricas.
metrics_scratch = None
y_true_scratch, y_pred_scratch = None, None
class_acc_scratch = None


## Bloque C — Transfer Learning con ResNet18


## Ejercicio 12 — Preparar datos compatibles con ImageNet

**Objetivo:** adaptar CIFAR-10 a un modelo preentrenado.

La ResNet18 fue preentrenada con imágenes y estadísticas de ImageNet. Por ello:

- redimensiona a 96×96 en modo rápido y 128×128 en modo completo;
- normaliza con media/desviación de ImageNet;
- usa `RandomResizedCrop` y volteo horizontal solo en entrenamiento.

Crea datasets y loaders usando **los mismos índices** del modelo anterior.


In [ ]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
TRANSFER_IMAGE_SIZE = 96 if FAST_MODE else 128

# TODO: crea train_transform_tl y eval_transform_tl.
train_transform_tl = None
eval_transform_tl = None

# TODO: crea datasets Subset y DataLoaders con los mismos índices.
train_tl_ds = None
val_tl_ds = None
test_tl_ds = None
train_tl_loader = None
val_tl_loader = None
test_tl_loader = None

tl_images_batch, tl_labels_batch = next(iter(train_tl_loader))
print("Lote para ResNet18:", None, None)


## Ejercicio 13 — Cargar ResNet18 y congelar el extractor

**Objetivo:** convertir una red de ImageNet en un clasificador de CIFAR-10.

1. Carga `resnet18(weights=ResNet18_Weights.DEFAULT)`.
2. Congela todos sus parámetros.
3. Sustituye `fc` por `Dropout(0.2) → Linear(512,10)`.
4. Comprueba que solo la nueva cabeza sea entrenable.
5. Ejecuta un `forward` con cuatro imágenes.


In [ ]:
# TODO: carga los pesos preentrenados.
weights = None
transfer_model = None

# TODO: congela todos los parámetros existentes.

# TODO: reemplaza fc por una cabeza para 10 clases.

transfer_model = transfer_model.to(DEVICE)

# TODO: realiza un forward con cuatro imágenes.
transfer_logits = None

print("Salida:", None)
print("Parámetros totales:", None)
print("Parámetros entrenables:", None)


## Ejercicio 14 — Entrenar únicamente la nueva cabeza

**Objetivo:** realizar la primera etapa de transferencia.

1. Optimiza solo los parámetros con `requires_grad=True`.
2. Usa `AdamW(lr=3e-3, weight_decay=1e-4)` y scheduler cosenoidal.
3. Entrena durante `HEAD_EPOCHS` manteniendo las estadísticas de BatchNorm congeladas.
4. Grafica las curvas.

**Pregunta breve:** ¿por qué puede usarse una tasa de aprendizaje relativamente alta en la nueva cabeza?


In [ ]:
# TODO: crea optimizador y scheduler solo con parámetros entrenables.
head_optimizer = None
head_scheduler = None

# TODO: entrena con freeze_bn=True.
history_head, time_head = None, None

plot_history(history_head, "Transfer learning — cabeza")


## Ejercicio 15 — Fine-tuning del último bloque

**Objetivo:** adaptar características de alto nivel sin destruir las representaciones generales.

1. Descongela únicamente `layer4`.
2. Usa tasas diferenciadas: `1e-4` para `layer4` y `5e-4` para `fc`.
3. Entrena `FINETUNE_EPOCHS` manteniendo todavía congeladas las estadísticas de BatchNorm.
4. Une los historiales de ambas etapas y grafica el proceso completo.

**Pregunta clave:** ¿por qué la tasa del backbone es menor que la de la cabeza?


In [ ]:
# TODO: descongela layer4.

# TODO: crea AdamW con dos grupos de parámetros y scheduler.
finetune_optimizer = None
finetune_scheduler = None

# TODO: entrena la segunda etapa.
history_finetune, time_finetune = None, None

def concatenate_histories(*histories):
    # TODO: concatena cada lista de métricas.
    pass

history_transfer = concatenate_histories(history_head, history_finetune)
plot_history(history_transfer, "ResNet18 — cabeza + fine-tuning")


## Bloque D — Comparación de enfoques


## Ejercicio 16 — Evaluar y comparar ambos modelos

**Objetivo:** contrastar desempeño y costo, no solo exactitud.

1. Evalúa ResNet18 en el mismo subconjunto de prueba.
2. Compara con la CNN propia:
   - pérdida y exactitud;
   - tiempo de entrenamiento;
   - parámetros totales y entrenables al final;
   - exactitud por clase.
3. Dibuja una barra de exactitud global y barras agrupadas por clase.
4. Interpreta cuándo compensa utilizar transferencia.


In [ ]:
# TODO: evalúa ResNet18 y calcula accuracy por clase.
metrics_transfer = None
y_true_transfer, y_pred_transfer = None, None
class_acc_transfer = None
time_transfer = None

# TODO: crea e imprime comparison.
comparison = None

# TODO: grafica accuracy global y accuracy por clase.


## Ejercicio 17 — Analizar mejoras y guardar modelos

**Objetivo:** estudiar casos concretos y conservar los resultados.

1. Encuentra imágenes que la CNN propia clasificó mal pero ResNet18 acertó.
2. Muestra hasta 10 con etiqueta real y ambas predicciones.
3. Guarda dos checkpoints con `state_dict`, clases y configuración.
4. Explica por qué conviene guardar metadatos además de los pesos.


In [ ]:
# TODO: localiza casos corregidos por transferencia y visualízalos con raw_test.

# TODO: guarda ambos checkpoints con pesos y metadatos.
scratch_checkpoint = Path("cifar10_cnn_scratch.pt")
transfer_checkpoint = Path("cifar10_resnet18_transfer.pt")


## Cierre

Has completado dos estrategias para CIFAR-10:

1. **CNN desde cero:** todos los filtros se aprenden con CIFAR-10.
2. **Transfer learning:** se reutiliza ResNet18, se entrena una cabeza nueva y luego se ajusta `layer4` con una tasa menor.

### Qué debe incluir tu conclusión

- cuál modelo obtuvo mayor exactitud global y por clase;
- cuál necesitó menos tiempo en tu hardware;
- cómo influyeron la cantidad de datos y las épocas;
- si la mejora justifica el tamaño y costo de ResNet18;
- por qué una comparación seria debe repetirse con varias semillas.

### Próximos pasos

- probar `EfficientNet-B0` o `MobileNetV3`;
- usar AMP para entrenamiento en precisión mixta;
- aplicar *early stopping* y búsqueda de hiperparámetros;
- visualizar activaciones o usar Grad-CAM para interpretar decisiones.
